# 08 — MongoDB: Agregações e Índices

**Objetivos:** construir pipelines e compreender índices e explain no MongoDB real

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia Técnico](./reference_materials/08_mongodb_agregacoes_indices.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_mongodb_agregacoes_indices.pdf)
- [Documentação da área](./README.pt-BR.md)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## Aggregation pipeline

Etapas como `$match`, `$unwind`, `$group` e `$sort` transformam documentos em resultados agregados. A ordem pode afetar uso de índices e volume processado.

In [2]:
import json,os
usuarios=json.loads((DATA/'usuarios_sinteticos.json').read_text(encoding='utf8'))
pipeline=[{'$unwind':'$interesses'},{'$group':{'_id':'$interesses','quantidade':{'$sum':1}}},{'$sort':{'quantidade':-1,'_id':1}}]
print(json.dumps(pipeline,ensure_ascii=False,indent=2))

[
  {
    "$unwind": "$interesses"
  },
  {
    "$group": {
      "_id": "$interesses",
      "quantidade": {
        "$sum": 1
      }
    }
  },
  {
    "$sort": {
      "quantidade": -1,
      "_id": 1
    }
  }
]


In [3]:
# Pré-requisito opcional: serviço MongoDB local e RUN_LIVE_DB=1.
# Nenhuma conexão será feita ao executar Run All no modo padrão.
import os
if os.getenv('RUN_LIVE_DB') == '1':
    from pymongo import MongoClient
    with MongoClient('mongodb://127.0.0.1:27017',serverSelectionTimeoutMS=5000) as cliente:
        print('Servidor alcançável:',cliente.admin.command('ping'))
else:
    print('Diagnóstico remoto ignorado (modo seguro, sem MongoDB).')
print('Usuários definidos:', 'usuarios' in globals())
print('Pipeline definido:', 'pipeline' in globals())

Diagnóstico remoto ignorado (modo seguro, sem MongoDB).
Usuários definidos: True
Pipeline definido: True


In [4]:
import os
print(os.getenv("RUN_LIVE_DB"))

None


In [5]:
# A execução local não ativa conexões automaticamente.
# Para laboratórios opcionais, configure RUN_LIVE_DB=1 antes de iniciar o kernel,
# depois de verificar que a instância é EXCLUSIVAMENTE de estudos.
import os
print('Laboratório externo:', 'habilitado' if os.getenv('RUN_LIVE_DB') == '1' else 'desabilitado (padrão seguro)')

Laboratório externo: desabilitado (padrão seguro)


In [6]:
# IMPORTANTE: só execute em instância de laboratório isolada.
# Cria coleção NOVA e exclusiva; nunca exclui coleções preexistentes.
import os
if os.getenv('RUN_LIVE_DB') == '1':
    from uuid import uuid4
    from pymongo import MongoClient
    with MongoClient('mongodb://127.0.0.1:27017',serverSelectionTimeoutMS=3000) as cli:
        cli.admin.command('ping')
        nome = 'aula08_' + uuid4().hex[:12]
        col = cli['ep04_laboratorio'][nome]
        col.insert_many(usuarios)
        col.create_index([('cidade',1),('ativo',1)])
        print('Coleção criada nesta sessão:',nome)
        print('Agregação real:', list(col.aggregate(pipeline)))
        print('Índices:', list(col.index_information()))
else:
    print('Laboratório MongoDB não executado (RUN_LIVE_DB diferente de 1).')

Laboratório MongoDB não executado (RUN_LIVE_DB diferente de 1).


## Prática real — pipeline e índice

Ative MongoDB local e `RUN_LIVE_DB=1`. Para evitar depender de uma execução prévia, este notebook prepara sua própria coleção de treinamento.

## Exercício resolvido — interesse mais comum

Compare o pipeline com a contagem local, que atua apenas como verificação de expectativa, e não como emulação da engine.

In [7]:
from collections import Counter
cont=Counter(t for user in usuarios for t in user['interesses']);print(cont.most_common());assert cont['dados']==3

[('dados', 3), ('livros', 2), ('esportes', 2), ('café', 1), ('música', 1), ('cinema', 1)]


## Interpretação de desempenho

Índices aceleram algumas leituras, mas custam espaço e escrita; `explain()` deve ser analisado com cardinalidade e dados reais, não com apenas seis documentos.

## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Pipeline e índices adequados

Em MongoDB, o desempenho de `$match`, `$group` e `$unwind` depende da ordem e da seletividade. Avalie com `explain()` quando o servidor estiver disponível e não interprete um índice criado como garantia de aceleração. Diferencie a saída de agregação real da verificação em Python.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [8]:
from collections import Counter
contagem=Counter(i for u in usuarios for i in u.get('interesses',[]))
assert sum(contagem.values())==sum(len(u.get('interesses',[])) for u in usuarios)
print('Agregação de referência:',contagem.most_common(5))

Agregação de referência: [('dados', 3), ('livros', 2), ('esportes', 2), ('café', 1), ('música', 1)]


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [9]:
pd.DataFrame(sorted(contagem.items()),columns=['interesse','quantidade']).to_csv(OUT / '08_interesses_contagem_local.csv',index=False)
print('Exportado: outputs/08_interesses_contagem_local.csv')

Exportado: outputs/08_interesses_contagem_local.csv
